# 08 — Human Baseline: Episode Log & Itinerary Builder

**Purpose:** Plan a trip as a human expert using the travel world tools.
Every search and booking is automatically logged as a `ReActStep`, producing
an `EpisodeLog` in the same format as agent-generated episodes.
The resulting log is evaluated with the same metrics used to score the agent.

**Workflow:**
1. Run **Setup** (cell 1.1 imports, then one of Options A / B / C to pick a request).
2. Run **1.2** to review constraints and **1.3** to start the session.
3. Discover the world with `get_available_routes()` — city IDs are auto-matched.
4. Search and book flights, hotels, attractions, and events.
5. Add each booking to the itinerary with `add_*_to_itinerary()` helpers.
6. Check progress with `show_itinerary()` / `show_budget()` at any time.
7. Call `session.finalize()` and `session.save()` when done.
8. Run **Section 11** to score your plan and compare against the agent.

> **Tip:** Each tool cell is independent — re-run any cell to retry with different arguments.
> Failed tool calls print an error and return `None`; the session continues normally.

---
## 1. Setup

In [1]:
import sys
from pathlib import Path

# Add project src to path (works whether you launch from repo root or notebooks/)
repo_root = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import json
import pandas as pd

from optimized_llm_planning_memory.core.models import UserRequest
from optimized_llm_planning_memory.evaluation.human_baseline import (
    HumanPlanningSession,
    display_routes,
    display_flights,
    display_hotels,
    display_attractions,
    display_restaurants,
    display_events,
    display_scores,
)

print('Imports OK')

Imports OK


### 1.1 Pick a user request

Choose **one** of the three options below (run only that cell, skip the others):

| Option | When to use |
|--------|-------------|
| **A — from agent episode** *(recommended)* | You want to plan the same trip the agent ran, enabling direct score comparison |
| **B — from a request JSON file** | Replay a specific saved request without an existing agent episode |
| **C — create inline** | Ad-hoc / new scenario not yet saved to disk |

> After running your chosen option, `user_request` and `WORLD_SEED` will be set.
> Proceed to **1.2** to print constraints, then **1.3** to start the session.

In [2]:
# ── OPTION A: Load user request from a saved agent episode ──────────────────
# This is the PRIMARY path: plan the same trip the agent ran so scores compare
# directly. The user_request and world_seed are extracted from the episode.

_app_path = str(repo_root)
if _app_path not in sys.path:
    sys.path.insert(0, _app_path)

from app.utils.data_loader import load_episodes as _load_episodes

_all_episodes = _load_episodes()
if not _all_episodes:
    print('⚠️  No saved episodes found in outputs/episodes/')
    print('   Run the agent first (scripts/run_episode.py or the Streamlit UI),')
    print('   then come back and use Option A.')
else:
    _rows = []
    for _i, _ep in enumerate(_all_episodes):
        _req = getattr(_ep, 'user_request', None)
        _rows.append({
            'idx': _i,
            'episode_id': _ep.episode_id[:16] + '...',
            'mode': _ep.agent_mode,
            'request_id': _ep.request_id,
            'raw_text': (_req.raw_text[:60] + '...') if _req else '—',
            'steps': _ep.total_steps,
            'reward': round(_ep.reward_components.total_reward, 3),
            'world_seed': _ep.world_seed,
            'created': _ep.created_at[:19].replace('T', ' '),
        })
    display(pd.DataFrame(_rows).set_index('idx'))


,episode_id,mode,request_id,raw_text,steps,reward,world_seed,created
idx,,,,,,,,
0,a299e33c-0b58-43...,compressor,TEMPLATE,Plan a 7-day trip from Aeloria to Brindor and ...,0,0.0,None,2026-05-02 22:09:25
1,e0236455-e27e-41...,compressor,TEMPLATE,Plan a 7-day trip from Aeloria to Brindor and ...,29,0.0,None,2026-05-02 22:23:52


In [3]:
# ── SELECT EPISODE ──────────────────────────────────────────────────────────
# Set AGENT_EPISODE_IDX to the idx of the episode you want to compare against.
# Run Option A listing cell above first if this cell raises NameError.

AGENT_EPISODE_IDX = 1   # ← change to the idx shown in the table above

if '_all_episodes' not in dir() or not _all_episodes:
    print('⚠️  _all_episodes not found — run the Option A listing cell above first.')
else:
    _agent_ep        = _all_episodes[AGENT_EPISODE_IDX]
    AGENT_EPISODE_ID = _agent_ep.episode_id
    user_request     = _agent_ep.user_request
    WORLD_SEED       = _agent_ep.world_seed or 42   # fall back to 42 if not recorded
    OUTPUT_DIR       = repo_root / 'outputs' / 'human_episodes'

    print(f'Agent episode : {AGENT_EPISODE_ID}')
    print(f'World seed    : {WORLD_SEED}')
    print(f'Request ID    : {user_request.request_id}')
    print(f'Request       : {user_request.raw_text}')
    print(f'Budget        : ${user_request.budget_usd:,.2f}')
    print(f'Dates         : {user_request.start_date} → {user_request.end_date}')


Agent episode : e0236455-e27e-4133-ae45-9558fa695999
World seed    : 42
Request ID    : TEMPLATE
Request       : Plan a 7-day trip from Aeloria to Brindor and Caelum for 2 adults with a budget of $5000. We prefer boutique hotels, love art museums, and need vegetarian dining options.
Budget        : $5,000.00
Dates         : 2026-06-01 → 2026-06-07


In [42]:
# ── OPTION B: Load user request from a JSON file ─────────────────────────────
# Use when you have a request JSON but no existing agent episode to compare.
# AGENT_EPISODE_ID is set to None (no direct comparison available).

# ── CONFIGURE: change this path to the request you want to plan ──────────────
REQUEST_FILE = repo_root / 'data' / 'user_requests' / 'templates' / 'request_template.json'

# Optional: override any field before creating the session
# WORLD_SEED = 42  # set to match the agent episodes you want to compare against
WORLD_SEED = 42
OUTPUT_DIR = repo_root / 'outputs' / 'human_episodes'
# ─────────────────────────────────────────────────────────────────────────────

raw = json.loads(Path(REQUEST_FILE).read_text())
user_request = UserRequest.model_validate(raw)

print(f'Loaded request: {user_request.request_id}')
print(f'  {user_request.raw_text}')
print(f'  Budget: ${user_request.budget_usd:,.2f}')
print(f'  Dates : {user_request.start_date} → {user_request.end_date}')
print(f'  Hard constraints: {len(user_request.hard_constraints)}')
print(f'  Soft constraints: {len(user_request.soft_constraints)}')
AGENT_EPISODE_ID = None   # no agent episode to compare against


Loaded request: TEMPLATE
  Plan a 7-day trip from Aeloria to Brindor and Caelum for 1 adult with a budget of $2500. I prefer boutique hotels, love art museums, and need vegetarian dining options.
  Budget: $2,500.00
  Dates : 2026-06-01 → 2026-06-07
  Hard constraints: 5
  Soft constraints: 3


In [ ]:
# ── OPTION C: Create a new request inline ────────────────────────────────────
# Use for ad-hoc scenarios not yet saved as JSON or agent episodes.
# AGENT_EPISODE_ID is set to None (no direct comparison available).
# IMPORTANT: run only ONE of Options A, B, or C.

import uuid as _uuid
from optimized_llm_planning_memory.core.models import (
    Constraint, ConstraintType, ConstraintCategory, TravelerProfile
)

# ── Trip details ─────────────────────────────────────────────────────────────
RAW_TEXT = (
    "Plan a 5-day trip from Aeloria to Brindor for 2 adults with a budget of $3000. "
    "We love beach activities and need gluten-free dining options."
)
ORIGIN_CITY        = "Aeloria"
DESTINATION_CITIES = ["Brindor"]   # add more: ["Brindor", "Caelum"]
START_DATE_STR     = "2026-07-01"
END_DATE_STR       = "2026-07-05"
BUDGET_USD         = 3000.0

# ── Travelers ────────────────────────────────────────────────────────────────
traveler_profile = TravelerProfile(
    num_adults=2,
    num_children=0,
    dietary_restrictions=["gluten-free"],  # e.g. ["vegetarian"], ["vegan"], []
    accessibility_needs=[],
)

# ── Hard constraints (MUST satisfy) ──────────────────────────────────────────
import datetime as _dt
_num_days = (
    _dt.date.fromisoformat(END_DATE_STR) - _dt.date.fromisoformat(START_DATE_STR)
).days + 1

hard_constraints = [
    Constraint(
        constraint_id=f"hc-budget-{_uuid.uuid4().hex[:6]}",
        constraint_type=ConstraintType.HARD,
        category=ConstraintCategory.BUDGET,
        description=f"Total trip cost must not exceed ${BUDGET_USD:,.0f} USD.",
        value=BUDGET_USD, unit='USD',
    ),
    Constraint(
        constraint_id=f"hc-dates-{_uuid.uuid4().hex[:6]}",
        constraint_type=ConstraintType.HARD,
        category=ConstraintCategory.DATE,
        description=f"Trip must start on {START_DATE_STR} and end on {END_DATE_STR}.",
        value=f"{START_DATE_STR} to {END_DATE_STR}", unit="date_range",
    ),
    Constraint(
        constraint_id=f"hc-duration-{_uuid.uuid4().hex[:6]}",
        constraint_type=ConstraintType.HARD,
        category=ConstraintCategory.DURATION,
        description=f"Trip must be exactly {_num_days} days long.",
        value=_num_days, unit='days',
    ),
    Constraint(
        constraint_id=f"hc-group-{_uuid.uuid4().hex[:6]}",
        constraint_type=ConstraintType.HARD,
        category=ConstraintCategory.GROUP,
        description=(
            f"Party of {traveler_profile.total_travelers} "
            f"({traveler_profile.num_adults} adult(s)); all pricing must accommodate the full group."
        ),
        value=traveler_profile.total_travelers, unit='people',
    ),
    # ── Add or remove activity hard constraints below ─────────────────────────
    Constraint(
        constraint_id=f"hc-activity-{_uuid.uuid4().hex[:6]}",
        constraint_type=ConstraintType.HARD,
        category=ConstraintCategory.ACTIVITY,
        description="Itinerary must include at least one beach activity.",
        value="beach", unit=None,
    ),
]

# ── Soft constraints (PREFER to satisfy) ─────────────────────────────────────
soft_constraints = [
    Constraint(
        constraint_id=f"sc-accommodation-{_uuid.uuid4().hex[:6]}",
        constraint_type=ConstraintType.SOFT,
        category=ConstraintCategory.ACCOMMODATION,
        description="Prefer 3-star hotels or above.",
        value=3, unit='min_stars',
    ),
]

PREFERENCES = ['beach', 'local cuisine', 'water sports']

# ── Build the UserRequest ─────────────────────────────────────────────────────
user_request = UserRequest(
    request_id=f"CUSTOM_{_uuid.uuid4().hex[:8].upper()}",
    raw_text=RAW_TEXT,
    origin_city=ORIGIN_CITY,
    destination_cities=DESTINATION_CITIES,
    start_date=START_DATE_STR,
    end_date=END_DATE_STR,
    budget_usd=BUDGET_USD,
    traveler_profile=traveler_profile,
    hard_constraints=hard_constraints,
    soft_constraints=soft_constraints,
    preferences=PREFERENCES,
)

print(f'Created request: {user_request.request_id}')
print(f'  {user_request.raw_text}')
print(f'  Budget: ${user_request.budget_usd:,.2f}')
print(f'  Dates : {user_request.start_date} → {user_request.end_date}')
print(f'  Hard constraints: {len(user_request.hard_constraints)}')
print(f'  Soft constraints: {len(user_request.soft_constraints)}')

WORLD_SEED       = 42    # fixed seed for reproducibility
AGENT_EPISODE_ID = None  # no agent episode to compare against
OUTPUT_DIR       = repo_root / 'outputs' / 'human_episodes'


### 1.2 Print the constraints (what you must satisfy)

In [4]:
print('HARD CONSTRAINTS (must satisfy):')
for c in user_request.hard_constraints:
    print(f'  [{c.category.value.upper()}]  {c.description}')

print('\nSOFT CONSTRAINTS (prefer to satisfy):')
for c in user_request.soft_constraints:
    print(f'  [{c.category.value.upper()}]  {c.description}')

print('\nPREFERENCES:')
for p in user_request.preferences:
    print(f'  • {p}')

HARD CONSTRAINTS (must satisfy):
  [BUDGET]  Total trip cost must not exceed $5000
  [DATE]  Trip must start on 2026-06-01 and end on 2026-06-07

SOFT CONSTRAINTS (prefer to satisfy):
  [ACCOMMODATION]  Prefer boutique hotels over large chains
  [PREFERENCE]  Vegetarian dining options at all meal stops

PREFERENCES:
  • art museums
  • walking tours
  • local cuisine


### 1.3 Create the planning session

In [5]:
# WORLD_SEED and OUTPUT_DIR are set by whichever Option (A/B/C) you ran above.
session = HumanPlanningSession(
    user_request=user_request,
    seed=WORLD_SEED,
    output_dir=OUTPUT_DIR,
    world_config={'num_cities_per_region': 3, 'num_regions': 1},
)


╔══════════════════════════════════════════════════════════╗
║  Human Planning Session                                  ║
╠══════════════════════════════════════════════════════════╣
║  episode_id  : 0759f9132644                              ║
║  request_id  : TEMPLATE                                  ║
║  world seed  : 42                                        ║
║  budget      : $5,000.00                                 ║
║  dates       : 2026-06-01 → 2026-06-07                  ║
╚══════════════════════════════════════════════════════════╝

Request: Plan a 7-day trip from Aeloria to Brindor and Caelum for 2 adults with a budget of $5000. We prefer boutique hotels, love art museums, and need vegetarian dining options.



---
## 2. Discover the World

Always run this first — it reveals the city IDs required by every other tool.

In [6]:
routes = session.get_available_routes(
    thought='First I need to discover available cities and their IDs.'
)
display_routes(routes)

,city_name,city_id,connects_to
1,Aeloria,city_world_42_20260504_010845_0000,
2,Brindor,city_world_42_20260504_010845_0001,
3,Caelum,city_world_42_20260504_010845_0002,


In [7]:
# ── AUTO-MATCH city IDs by name from the routes table above ─────────────────
# The simulator assigns city names that match your origin/destination names.
# Auto-fill looks these up by name so you don't need to copy IDs manually.

_name_to_id = {r['city_name'].lower(): r['city_id'] for r in (routes or [])}

ORIGIN_ID = _name_to_id.get(user_request.origin_city.lower(), '')
DEST_1_ID = (
    _name_to_id.get(user_request.destination_cities[0].lower(), '')
    if user_request.destination_cities else ''
)
DEST_2_ID = (
    _name_to_id.get(user_request.destination_cities[1].lower(), '')
    if len(user_request.destination_cities) > 1 else ''
)

# ── Override manually if auto-fill didn't work ────────────────────────────
# ORIGIN_ID = 'city_world_42_...'   # copy from routes table above
# DEST_1_ID = 'city_world_42_...'
# DEST_2_ID = ''   # leave '' if only one destination

START_DATE = user_request.start_date
END_DATE   = user_request.end_date
PASSENGERS = user_request.traveler_profile.total_travelers

print(f'Origin : {ORIGIN_ID!r}')
print(f'Dest 1 : {DEST_1_ID!r}')
print(f'Dest 2 : {DEST_2_ID!r} (empty = only one destination)')
print(f'Dates  : {START_DATE} → {END_DATE}  |  Passengers: {PASSENGERS}')
if not ORIGIN_ID or not DEST_1_ID:
    print('\n⚠️  Auto-match failed — set ORIGIN_ID and DEST_1_ID in the override block above.')


Origin : 'city_world_42_20260504_010845_0000'
Dest 1 : 'city_world_42_20260504_010845_0001'
Dest 2 : 'city_world_42_20260504_010845_0002' (empty = only one destination)
Dates  : 2026-06-01 → 2026-06-07  |  Passengers: 2


---
## 3. Flights

Search → inspect results → select the best option → add to itinerary.

### 3.1 Outbound flight (Origin → Destination 1)

In [8]:
flights_out = session.search_flights(
    thought=f'Searching for flights from {ORIGIN_ID} to {DEST_1_ID} on {START_DATE}.',
    origin_city_id=ORIGIN_ID,
    destination_city_id=DEST_1_ID,
    departure_date=START_DATE,
    passengers=PASSENGERS,
)
display_flights(flights_out)

,edge_id,origin_city_id,destination_city_id,departure_datetime,arrival_datetime,total_price
1,edge_world_42_20260504_010845_0008,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T09:00:00,2026-06-01T12:10:00,950.87
2,edge_world_42_20260504_010845_0011,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T15:00:00,2026-06-01T18:05:00,1004.52
3,edge_world_42_20260504_010845_0066,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T09:00:00,2026-06-01T15:23:00,1051.08
4,edge_world_42_20260504_010845_0009,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T11:00:00,2026-06-01T14:35:00,1069.47
5,edge_world_42_20260504_010845_0007,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T07:00:00,2026-06-01T10:05:00,1180.76
6,edge_world_42_20260504_010845_0013,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T19:00:00,2026-06-01T22:25:00,1187.82
7,edge_world_42_20260504_010845_0014,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T21:00:00,2026-06-02T00:10:00,1188.08
8,edge_world_42_20260504_010845_0010,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T13:00:00,2026-06-01T16:25:00,1212.12
9,edge_world_42_20260504_010845_0068,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T21:00:00,2026-06-02T03:23:00,1217.86
10,edge_world_42_20260504_010845_0012,city_world_42_20260504_010845_0000,city_world_42_20260504_010845_0001,2026-06-01T17:00:00,2026-06-01T20:25:00,1291.85


In [9]:
# ── SELECT AND BOOK ───────────────────────────────────────────────────────────
# Copy edge_id from the table above (row [1] is cheapest).

FLIGHT_OUT_EDGE_ID = flights_out[0]['edge_id'] if flights_out else ''

booking_flight_out = session.select_flight(
    thought=f'Taking the cheapest outbound flight on {START_DATE}.',
    edge_id=FLIGHT_OUT_EDGE_ID,
)
print(booking_flight_out)

{'booking_id': 'FLT-50F91BE1', 'edge_id': 'edge_world_42_20260504_010845_0008', 'origin_city_name': 'city_world_42_20260504_010845_0000', 'destination_city_name': 'city_world_42_20260504_010845_0001', 'departure_datetime': '2026-06-01T09:00:00', 'arrival_datetime': '2026-06-01T12:10:00', 'total_price': 950.87, 'status': 'selected', 'confirmation_datetime': '2026-05-04T01:14:17.175128+00:00'}


In [10]:
# Add flight to the itinerary
if booking_flight_out:
    session.add_flight_to_itinerary(
        date_str=START_DATE,
        booking=booking_flight_out,
        thought=f'Outbound leg: {ORIGIN_ID} → {DEST_1_ID} on {START_DATE}.',
    )

  ✅ Outbound leg: city_world_42_20260504_010845_0000 → city_world_42_20260504_010845_0001 on 2026-06-01.


### 3.2 Inter-destination flight (Destination 1 → Destination 2)

*Skip this section if you only have one destination.*

In [11]:
# ── ADJUST: set the date you want to fly to destination 2 ────────────────────
INTER_DATE = '2026-06-04'   # e.g. '2026-06-04'

if DEST_2_ID and INTER_DATE:
    flights_inter = session.search_flights(
        thought=f'Searching flights from {DEST_1_ID} to {DEST_2_ID} on {INTER_DATE}.',
        origin_city_id=DEST_1_ID,
        destination_city_id=DEST_2_ID,
        departure_date=INTER_DATE,
        passengers=PASSENGERS,
    )
    display_flights(flights_inter)
else:
    print('Skipped (set DEST_2_ID and INTER_DATE to enable this cell)')
    flights_inter = []

,edge_id,origin_city_id,destination_city_id,departure_datetime,arrival_datetime,total_price
1,edge_world_42_20260504_010845_0041,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T15:00:00,2026-06-04T17:00:00,999.73
2,edge_world_42_20260504_010845_0038,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T09:00:00,2026-06-04T11:50:00,1084.49
3,edge_world_42_20260504_010845_0077,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T07:00:00,2026-06-04T14:15:00,1141.11
4,edge_world_42_20260504_010845_0037,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T07:00:00,2026-06-04T09:35:00,1154.97
5,edge_world_42_20260504_010845_0039,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T11:00:00,2026-06-04T13:20:00,1179.47
6,edge_world_42_20260504_010845_0043,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T19:00:00,2026-06-04T21:50:00,1253.65
7,edge_world_42_20260504_010845_0075,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T19:00:00,2026-06-05T02:15:00,1292.01
8,edge_world_42_20260504_010845_0040,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T13:00:00,2026-06-04T15:45:00,1297.77
9,edge_world_42_20260504_010845_0042,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T17:00:00,2026-06-04T19:35:00,1436.57
10,edge_world_42_20260504_010845_0044,city_world_42_20260504_010845_0001,city_world_42_20260504_010845_0002,2026-06-04T21:00:00,2026-06-04T23:00:00,1439.44


In [12]:
if flights_inter:
    booking_flight_inter = session.select_flight(
        thought=f'Selecting inter-destination flight on {INTER_DATE}.',
        edge_id=flights_inter[0]['edge_id'],
    )
    if booking_flight_inter:
        session.add_flight_to_itinerary(
            date_str=INTER_DATE,
            booking=booking_flight_inter,
            thought=f'Inter-destination leg: {DEST_1_ID} → {DEST_2_ID} on {INTER_DATE}.',
        )

  ✅ Inter-destination leg: city_world_42_20260504_010845_0001 → city_world_42_20260504_010845_0002 on 2026-06-04.


### 3.3 Return flight

In [13]:
# ── ADJUST: which city are you flying home from? ──────────────────────────────
RETURN_FROM_ID = DEST_2_ID if DEST_2_ID else DEST_1_ID

flights_return = session.search_flights(
    thought=f'Searching return flights from {RETURN_FROM_ID} to {ORIGIN_ID} on {END_DATE}.',
    origin_city_id=RETURN_FROM_ID,
    destination_city_id=ORIGIN_ID,
    departure_date=END_DATE,
    passengers=PASSENGERS,
)
display_flights(flights_return)

,edge_id,origin_city_id,destination_city_id,departure_datetime,arrival_datetime,total_price
1,edge_world_42_20260504_010845_0051,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T15:00:00,2026-06-07T17:20:00,1136.78
2,edge_world_42_20260504_010845_0048,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T09:00:00,2026-06-07T12:15:00,1236.49
3,edge_world_42_20260504_010845_0079,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T07:00:00,2026-06-07T14:36:00,1271.13
4,edge_world_42_20260504_010845_0049,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T11:00:00,2026-06-07T13:30:00,1366.90
5,edge_world_42_20260504_010845_0047,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T07:00:00,2026-06-07T09:15:00,1439.81
6,edge_world_42_20260504_010845_0050,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T13:00:00,2026-06-07T15:50:00,1482.44
7,edge_world_42_20260504_010845_0054,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T21:00:00,2026-06-07T23:20:00,1494.29
8,edge_world_42_20260504_010845_0053,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T19:00:00,2026-06-07T21:30:00,1507.10
9,edge_world_42_20260504_010845_0078,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T19:00:00,2026-06-08T02:36:00,1532.90
10,edge_world_42_20260504_010845_0052,city_world_42_20260504_010845_0002,city_world_42_20260504_010845_0000,2026-06-07T17:00:00,2026-06-07T19:40:00,1566.16


In [14]:
if flights_return:
    booking_flight_return = session.select_flight(
        thought=f'Taking the return flight on {END_DATE}.',
        edge_id=flights_return[0]['edge_id'],
    )
    if booking_flight_return:
        session.add_flight_to_itinerary(
            date_str=END_DATE,
            booking=booking_flight_return,
            thought=f'Return leg: {RETURN_FROM_ID} → {ORIGIN_ID} on {END_DATE}.',
        )

  ✅ Return leg: city_world_42_20260504_010845_0002 → city_world_42_20260504_010845_0000 on 2026-06-07.


---
## 4. Hotels

Book accommodation for each destination. Tip: set `max_price_per_night` to
`session.budget_remaining / remaining_nights` to filter out unaffordable options.

### 4.1 Hotel at Destination 1

In [15]:
# ── ADJUST: check-in and check-out dates for this destination ─────────────────
HOTEL_1_CHECK_IN  = START_DATE          # first night at destination 1
HOTEL_1_CHECK_OUT = INTER_DATE if INTER_DATE else END_DATE

hotels_1 = session.search_hotels(
    thought=f'Searching hotels in {DEST_1_ID} for {HOTEL_1_CHECK_IN} → {HOTEL_1_CHECK_OUT}.',
    city_id=DEST_1_ID,
    check_in=HOTEL_1_CHECK_IN,
    check_out=HOTEL_1_CHECK_OUT,
    guests=PASSENGERS,
    max_price_per_night=session.budget_remaining / max(1, 3),  # rough estimate
)
display_hotels(hotels_1)

,hotel_id,name,star_rating,price_per_night,total_cost
1,hotel_world_42_20260504_010845_0285,The Brindor Plaza,1.0,48.41,145.23
2,hotel_world_42_20260504_010845_0429,The Cultural Mile Inn,1.0,51.03,153.09
3,hotel_world_42_20260504_010845_0291,The Hilltop Inn,1.0,59.33,177.99
4,hotel_world_42_20260504_010845_0336,Brindor Grand,1.0,65.47,196.41
5,hotel_world_42_20260504_010845_0340,Brindor Grand,1.0,67.06,201.17
6,hotel_world_42_20260504_010845_0289,Brindor Residences,2.0,112.67,338.01
7,hotel_world_42_20260504_010845_0431,The Cultural Mile Inn,2.0,124.37,373.12
8,hotel_world_42_20260504_010845_0286,Brindor Suites,2.0,124.40,373.21
9,hotel_world_42_20260504_010845_0430,The Brindor Plaza,3.0,207.45,622.36
10,hotel_world_42_20260504_010845_0433,Brindor Grand,3.0,210.93,632.80


In [17]:
if hotels_1:
    HOTEL_1_ID = hotels_1[0]['hotel_id']   # pick from the table above

    booking_hotel_1 = session.book_hotel(
        thought=f'Booking the top-rated affordable hotel in {DEST_1_ID}.',
        hotel_id=HOTEL_1_ID,
        check_in=HOTEL_1_CHECK_IN,
        check_out=HOTEL_1_CHECK_OUT,
    )
    print(booking_hotel_1)

{'booking_id': 'dc3f8569-fc83-44b9-b817-ef79bd4bf919', 'hotel_id': 'hotel_world_42_20260504_010845_0285', 'hotel_name': 'The Brindor Plaza', 'check_in': '2026-06-01', 'check_out': '2026-06-04', 'num_nights': 3, 'event_id': '', 'event_name': '', 'quantity': 0, 'total_cost': 145.23, 'price_per_night': 48.41, 'session_id': '777b7a7e-ac34-4f6f-b6e6-18218fe4db92', 'status': 'confirmed'}


In [18]:
if hotels_1 and booking_hotel_1:
    session.add_hotel_to_itinerary(
        booking=booking_hotel_1,
        city=DEST_1_ID,
        # Provide these if the booking dict doesn't have them
        hotel_name=hotels_1[0].get('name', ''),
        star_rating=hotels_1[0].get('star_rating'),
        cost_per_night=hotels_1[0].get('price_per_night'),
        thought=f'Accommodation in {DEST_1_ID} for {HOTEL_1_CHECK_IN} → {HOTEL_1_CHECK_OUT}.',
    )

  ✅ Accommodation in city_world_42_20260504_010845_0001 for 2026-06-01 → 2026-06-04.


### 4.2 Hotel at Destination 2 (skip if only one destination)

In [19]:
HOTEL_2_CHECK_IN  = INTER_DATE if INTER_DATE else ''
HOTEL_2_CHECK_OUT = END_DATE

if DEST_2_ID and HOTEL_2_CHECK_IN:
    hotels_2 = session.search_hotels(
        thought=f'Searching hotels in {DEST_2_ID} for {HOTEL_2_CHECK_IN} → {HOTEL_2_CHECK_OUT}.',
        city_id=DEST_2_ID,
        check_in=HOTEL_2_CHECK_IN,
        check_out=HOTEL_2_CHECK_OUT,
        guests=PASSENGERS,
        max_price_per_night=session.budget_remaining / max(1, 3),
    )
    display_hotels(hotels_2)
else:
    print('Skipped')
    hotels_2 = []

,hotel_id,name,star_rating,price_per_night,total_cost
1,hotel_world_42_20260504_010845_0508,Hotel Cae10,1.0,79.00,237.00
2,hotel_world_42_20260504_010845_0504,Caelum Residences,1.0,79.96,239.88
3,hotel_world_42_20260504_010845_0547,Caelum Grand,2.0,146.68,440.05
4,hotel_world_42_20260504_010845_0550,Caelum Grand,2.0,160.19,480.57
5,hotel_world_42_20260504_010845_0512,Caelum Grand,2.0,171.53,514.58
6,hotel_world_42_20260504_010845_0473,Hotel Cae2,2.0,188.19,564.58
7,hotel_world_42_20260504_010845_0546,Caelum Grand,3.0,240.77,722.31
8,hotel_world_42_20260504_010845_0549,The Northgate Inn,3.0,248.01,744.03
9,hotel_world_42_20260504_010845_0548,Caelum Residences,3.0,251.97,755.91
10,hotel_world_42_20260504_010845_0635,Grand Greenfields,3.0,268.04,804.11


In [20]:
if hotels_2:
    HOTEL_2_ID = hotels_2[0]['hotel_id']
    booking_hotel_2 = session.book_hotel(
        thought=f'Booking hotel in {DEST_2_ID}.',
        hotel_id=HOTEL_2_ID,
        check_in=HOTEL_2_CHECK_IN,
        check_out=HOTEL_2_CHECK_OUT,
    )
    if booking_hotel_2:
        session.add_hotel_to_itinerary(
            booking=booking_hotel_2,
            city=DEST_2_ID,
            hotel_name=hotels_2[0].get('name', ''),
            star_rating=hotels_2[0].get('star_rating'),
            cost_per_night=hotels_2[0].get('price_per_night'),
            thought=f'Accommodation in {DEST_2_ID} for {HOTEL_2_CHECK_IN} → {HOTEL_2_CHECK_OUT}.',
        )

  ✅ Accommodation in city_world_42_20260504_010845_0002 for 2026-06-04 → 2026-06-07.


In [21]:
# Show budget so far
session.show_budget()

  Budget            : $  5,000.00
  Transport         : $  3,087.38
  Accommodation     : $    382.23
  Activities/Events : $      0.00
  ─────────────────────────────
  Spent             : $  3,469.61
  Remaining         : $  1,530.39


---
## 5. Activities & Attractions

Search for things to do in each destination. Add the ones you want to each day.
Duplicate or modify these cells for each activity.

### 5.1 Attractions at Destination 1

In [22]:
# Search all categories first to get an overview
attractions_1 = session.search_attractions(
    thought=f'Exploring what attractions are available in {DEST_1_ID}.',
    city_id=DEST_1_ID,
)
display_attractions(attractions_1)

,attraction_id,name,category,admission_fee,location
1,attraction_world_42_20260504_010845_0352,Central Market Park,park,,
2,attraction_world_42_20260504_010845_0297,Hilltop Historic Site,historic_site,,
3,attraction_world_42_20260504_010845_0447,Brindor Museum,museum,,
4,attraction_world_42_20260504_010845_0355,The Great Park of Brindor,park,,
5,attraction_world_42_20260504_010845_0262,The Great Museum of Brindor,museum,,
6,attraction_world_42_20260504_010845_0260,Brindor Historic Site,historic_site,,
7,attraction_world_42_20260504_010845_0263,Brindor Shopping,shopping,,
8,attraction_world_42_20260504_010845_0294,Brindor Beach,beach,,
9,attraction_world_42_20260504_010845_0261,The Great Nightlife of Brindor,nightlife,,
10,attraction_world_42_20260504_010845_0435,Brindor Cultural Site,cultural_site,,


In [24]:
# ── REQUIRED ACTIVITIES (auto-derived from hard constraints) ────────────────
# Extracts activity types required by hard constraints and searches for them.

required_act_constraints = [
    c for c in user_request.hard_constraints
    if c.category.value == 'activity'
]

if required_act_constraints:
    print('Required activities (hard constraints):')
    for c in required_act_constraints:
        print(f'  • {c.value!r}  — {c.description}')
else:
    print('No activity-type hard constraints — skip to Optional Activities below.')

required_results = {}
for _req in required_act_constraints:
    print(f'\nSearching {DEST_1_ID!r} for {_req.value!r}...')
    _results = session.search_attractions(
        thought=(
            f'Searching for {_req.value} in {DEST_1_ID} — '
            f'required by hard constraint: {_req.description}'
        ),
        city_id=DEST_1_ID,
        category=_req.value,
    )
    required_results[_req.value] = _results
    display_attractions(_results)


No activity-type hard constraints — skip to Optional Activities below.


In [26]:
# ── ADD REQUIRED ACTIVITIES ─────────────────────────────────────────────────
# Adds the first result for each required activity type found above.
# Adjust date_str and start_time to fit your day-by-day schedule.

from datetime import date as _date, timedelta

for _act_type, _results in required_results.items():
    if not _results:
        print(f'  ⚠️  No {_act_type!r} found in {DEST_1_ID} — try searching another city.')
        continue
    _chosen = _results[0]   # change index to pick a different option
    session.add_activity_to_itinerary(
        date_str=START_DATE,          # ← adjust to whichever day you want this
        city=DEST_1_ID,
        activity_id=_chosen.get('attraction_id', ''),
        activity_name=_chosen.get('name', _act_type.title()),
        location=_chosen.get('location', DEST_1_ID),
        start_time='09:00',
        duration_hours=_chosen.get('duration_hours', 3.0),
        cost_usd=_chosen.get('admission_fee', 0.0) * PASSENGERS,
        category=_act_type,
        thought=f'Adding {_act_type} activity — satisfies hard constraint.',
    )

# ── ADD OPTIONAL ACTIVITIES ───────────────────────────────────────────────
# Copy this pattern to add more activities (museums, markets, parks, etc.)
#
_more = session.search_attractions(
    thought='Looking for optional activities matching preferences.',
    city_id=DEST_1_ID,
    category='museum',   # try: 'park', 'market', 'museum', 'gallery'
)
display_attractions(_more)
#
_more_list = _more['result'] if isinstance(_more, dict) and 'result' in _more else (_more or [])
if _more_list:
    _day2 = str(_date.fromisoformat(START_DATE) + timedelta(days=1))
    session.add_activity_to_itinerary(
        date_str=_day2,
        city=DEST_1_ID,
        activity_id=_more_list[0]['attraction_id'],
        activity_name=_more_list[0]['name'],
        start_time='14:00',
        duration_hours=_more_list[0].get('duration_hours', 2.0),
        cost_usd=_more_list[0].get('admission_fee', 0.0) * PASSENGERS,
        category='museum',
        thought='Brindor Museum on June 2 afternoon.',
    )


print('Add more activities by copying and modifying the pattern above.')


,attraction_id,name,category,admission_fee,location
1,attraction_world_42_20260504_010845_0447,Brindor Museum,museum,,
2,attraction_world_42_20260504_010845_0262,The Great Museum of Brindor,museum,,
3,attraction_world_42_20260504_010845_0445,Cultural Mile Museum,museum,,
4,attraction_world_42_20260504_010845_0349,Central Market Museum,museum,,
5,attraction_world_42_20260504_010845_0254,Riverside Museum,museum,,
6,attraction_world_42_20260504_010845_0296,Brindor Museum,museum,,
7,attraction_world_42_20260504_010845_0401,Brindor Museum,museum,,


  ✅ Brindor Museum on June 2 afternoon.
Add more activities by copying and modifying the pattern above.


In [27]:
# Add second museum
if _more_list and len(_more_list) > 1:
    _day3 = str(_date.fromisoformat(START_DATE) + timedelta(days=2))
    session.add_activity_to_itinerary(
        date_str=_day3,
        city=DEST_1_ID,
        activity_id=_more_list[1]['attraction_id'],
        activity_name=_more_list[1]['name'],
        start_time='09:00',
        duration_hours=_more_list[1].get('duration_hours', 2.0),
        cost_usd=_more_list[1].get('admission_fee', 0.0) * PASSENGERS,
        category='museum',
        thought='Cultural Mile Museum on June 3 morning.',
    )



  ✅ Cultural Mile Museum on June 3 morning.


In [28]:
# Add cultural site
_cultural = session.search_attractions(
    thought='Looking for cultural sites in Brindor.',
    city_id=DEST_1_ID,
    category='cultural_site',
)
display_attractions(_cultural)
_cultural_list = _cultural['result'] if isinstance(_cultural, dict) else (_cultural or [])
if _cultural_list:
    _day2 = str(_date.fromisoformat(START_DATE) + timedelta(days=1))
    session.add_activity_to_itinerary(
        date_str=_day2,
        city=DEST_1_ID,
        activity_id=_cultural_list[0]['attraction_id'],
        activity_name=_cultural_list[0]['name'],
        start_time='10:00',
        duration_hours=_cultural_list[0].get('duration_hours', 2.0),
        cost_usd=_cultural_list[0].get('admission_fee', 0.0) * PASSENGERS,
        category='cultural_site',
        thought='Cultural site in Brindor on June 2 morning.',
    )

,attraction_id,name,category,admission_fee,location
1,attraction_world_42_20260504_010845_0435,Brindor Cultural Site,cultural_site,,
2,attraction_world_42_20260504_010845_0350,Central Market Cultural Site,cultural_site,,
3,attraction_world_42_20260504_010845_0259,Riverside Cultural Site,cultural_site,,
4,attraction_world_42_20260504_010845_0351,Central Market Cultural Site,cultural_site,,


  ✅ Cultural site in Brindor on June 2 morning.


In [44]:
# Add food market
_market = session.search_attractions(
    thought='Looking for food markets in Brindor.',
    city_id=DEST_1_ID,
    category='food_market',
)
display_attractions(_market)
_market_list = _market['result'] if isinstance(_market, dict) else (_market or [])
if _market_list:
    _day3 = str(_date.fromisoformat(START_DATE) + timedelta(days=2))
    session.add_activity_to_itinerary(
        date_str=_day3,
        city=DEST_1_ID,
        activity_id=_market_list[0]['attraction_id'],
        activity_name=_market_list[0]['name'],
        start_time='14:00',
        duration_hours=_market_list[0].get('duration_hours', 2.0),
        cost_usd=_market_list[0].get('admission_fee', 0.0) * PASSENGERS,
        category='food_market',
        thought='Food market in Brindor on June 3 afternoon.',
    )

,attraction_id,name,category,admission_fee,location
1,attraction_world_42_20260504_010845_0403,Harbor View Food Market,food_market,,
2,attraction_world_42_20260504_010845_0440,Brindor Food Market,food_market,,
3,attraction_world_42_20260504_010845_0298,Brindor Food Market,food_market,,
4,attraction_world_42_20260504_010845_0255,Riverside Food Market,food_market,,
5,attraction_world_42_20260504_010845_0258,The Great Food Market of Brindor,food_market,,


  ✅ Food market in Brindor on June 3 afternoon.


### 5.2 Attractions at Destination 2 (skip if only one destination)

In [29]:
if DEST_2_ID:
    attractions_2 = session.search_attractions(
        thought=f'Exploring attractions in {DEST_2_ID}.',
        city_id=DEST_2_ID,
    )
    display_attractions(attractions_2)
else:
    print('Skipped')
    attractions_2 = []

,attraction_id,name,category,admission_fee,location
1,attraction_world_42_20260504_010845_0560,Caelum Park,park,,
2,attraction_world_42_20260504_010845_0563,Northgate Park,park,,
3,attraction_world_42_20260504_010845_0638,Caelum Gallery,gallery,,
4,attraction_world_42_20260504_010845_0564,Northgate Food Market,food_market,,
5,attraction_world_42_20260504_010845_0554,The Great Landmark of Caelum,landmark,,
6,attraction_world_42_20260504_010845_0561,Northgate Food Market,food_market,,
7,attraction_world_42_20260504_010845_0556,The Great Nightlife of Caelum,nightlife,,
8,attraction_world_42_20260504_010845_0553,Caelum Nightlife,nightlife,,
9,attraction_world_42_20260504_010845_0552,The Great Shopping of Caelum,shopping,,
10,attraction_world_42_20260504_010845_0559,Northgate Nightlife,nightlife,,


In [ ]:
# Add gallery (close to art)
_attractions_2_list = attractions_2['result'] if isinstance(attractions_2, dict) and 'result' in attractions_2 else (attractions_2 or [])
_day5 = str(_date.fromisoformat(START_DATE) + timedelta(days=4))  # June 5
session.add_activity_to_itinerary(
    date_str=_day5,
    city=DEST_2_ID,
    activity_id=_attractions_2_list[2]['attraction_id'],
    activity_name=_attractions_2_list[2]['name'],
    start_time='09:00',
    duration_hours=_attractions_2_list[2].get('duration_hours', 2.0),
    cost_usd=_attractions_2_list[2].get('admission_fee', 0.0) * PASSENGERS,
    category='gallery',
    thought='Caelum Gallery on June 5 morning — satisfies art preference.',
)

  ✅ Caelum Gallery on June 5 morning — satisfies art preference.


In [ ]:
# Add walking tour
_day5 = str(_date.fromisoformat(START_DATE) + timedelta(days=4))  # June 5
session.add_activity_to_itinerary(
    date_str=_day5,
    city=DEST_2_ID,
    activity_id=_attractions_2_list[4]['attraction_id'],
    activity_name=_attractions_2_list[4]['name'],
    start_time='14:00',
    duration_hours=_attractions_2_list[4].get('duration_hours', 2.0),
    cost_usd=_attractions_2_list[4].get('admission_fee', 0.0) * PASSENGERS,
    category='landmark',
    thought='Landmark walking tour in Caelum on June 5 afternoon.',
)


  ✅ Landmark walking tour in Caelum on June 5 afternoon.


In [ ]:
# Add walking tour
_day6 = str(_date.fromisoformat(START_DATE) + timedelta(days=5))  # June 6
session.add_activity_to_itinerary(
    date_str=_day6,
    city=DEST_2_ID,
    activity_id=_attractions_2_list[0]['attraction_id'],
    activity_name=_attractions_2_list[0]['name'],
    start_time='09:00',
    duration_hours=_attractions_2_list[0].get('duration_hours', 2.0),
    cost_usd=_attractions_2_list[0].get('admission_fee', 0.0) * PASSENGERS,
    category='park',
    thought='Caelum Park on June 6 morning — walking tour.',
)


  ✅ Caelum Park on June 6 morning — walking tour.


In [ ]:
# Add local cuisine
_day6 = str(_date.fromisoformat(START_DATE) + timedelta(days=5))  # June 6
session.add_activity_to_itinerary(
    date_str=_day6,
    city=DEST_2_ID,
    activity_id=_attractions_2_list[3]['attraction_id'],
    activity_name=_attractions_2_list[3]['name'],
    start_time='14:00',
    duration_hours=_attractions_2_list[3].get('duration_hours', 2.0),
    cost_usd=_attractions_2_list[3].get('admission_fee', 0.0) * PASSENGERS,
    category='food_market',
    thought='Northgate Food Market on June 6 afternoon — local cuisine.',
)


  ✅ Northgate Food Market on June 6 afternoon — local cuisine.


---
## 6. Restaurants

Restaurants are informational — no booking required.
Add them as ActivityBookings with `category='restaurant'`.

In [34]:
# Derive filters from the request's dietary restrictions
_dietary = user_request.traveler_profile.dietary_restrictions
if _dietary:
    print(f'Dietary restrictions from request: {_dietary}')
    print('Tip: uncomment cuisine= below to filter restaurants accordingly.')
else:
    print('No dietary restrictions — showing all restaurants.')

restaurants_1 = session.search_restaurants(
    thought=(
        f'Looking for restaurants in {DEST_1_ID}'
        + (f' — dietary restrictions: {", ".join(_dietary)}.' if _dietary else '.')
    ),
    city_id=DEST_1_ID,
    # cuisine=_dietary[0] if _dietary else None,  # uncomment to filter by first restriction
    # max_avg_spend=30.0,                          # uncomment to cap per-person spend
)
display_restaurants(restaurants_1)


Dietary restrictions from request: ['vegetarian']
Tip: uncomment cuisine= below to filter restaurants accordingly.


,restaurant_id,name,cuisine,avg_spend_per_person
1,restaurant_world_42_20260504_010845_0266,Mediterranean Kitchen Bri3,,
2,restaurant_world_42_20260504_010845_0461,Cultural Mile Italian Bistro,,
3,restaurant_world_42_20260504_010845_0318,Chez Hilltop,,
4,restaurant_world_42_20260504_010845_0301,Casa Brin,,
5,restaurant_world_42_20260504_010845_0456,Maison Brin,,
6,restaurant_world_42_20260504_010845_0383,Casa Brin,,
7,restaurant_world_42_20260504_010845_0420,Chez Harbor View,,
8,restaurant_world_42_20260504_010845_0374,Maison Brin,,
9,restaurant_world_42_20260504_010845_0417,Harbor View Vietnamese Bistro,,
10,restaurant_world_42_20260504_010845_0414,Japanese Kitchen Bri7,,


In [35]:
# ── ADD A RESTAURANT VISIT ────────────────────────────────────────────────────
# Restaurants are not bookable — just record the visit as an activity.

# Add mediterranean- likely to have vegan options
if restaurants_1:
    chosen_resto = restaurants_1[0]
    avg_spend = chosen_resto.get('avg_spend_per_person', 25.0)

    from datetime import date as _date, timedelta
    day2 = str(_date.fromisoformat(START_DATE) + timedelta(days=1))

    session.add_activity_to_itinerary(
        date_str=day2,
        city=DEST_1_ID,
        activity_id=chosen_resto.get('restaurant_id', ''),
        activity_name=chosen_resto.get('name', 'Restaurant'),
        location=chosen_resto.get('location', DEST_1_ID),
        start_time='19:00',
        duration_hours=1.5,
        cost_usd=avg_spend * PASSENGERS,
        category='restaurant',
        thought=f'Dinner at a local restaurant{" — dietary: " + ", ".join(_dietary) if _dietary else ""}.',
    )

  ✅ Dinner at a local restaurant — dietary: vegetarian.


In [37]:
# Add Vietnamese restaurant
if restaurants_1:
    chosen_resto2 = restaurants_1[8]
    day3 = str(_date.fromisoformat(START_DATE) + timedelta(days=2))
    session.add_activity_to_itinerary(
        date_str=day3,
        city=DEST_1_ID,
        activity_id=chosen_resto2.get('restaurant_id', ''),
        activity_name=chosen_resto2.get('name', 'Restaurant'),
        location=chosen_resto2.get('location', DEST_1_ID),
        start_time='19:00',
        duration_hours=1.5,
        cost_usd=chosen_resto2.get('average_spend', 25.0) * PASSENGERS,
        category='restaurant',
        thought='Dinner at Vietnamese Bistro — vegetarian-friendly option.',
    )

  ✅ Dinner at Vietnamese Bistro — vegetarian-friendly option.


In [ ]:
# Now look for Caelum restauarants
restaurants_2 = session.search_restaurants(
    thought='Looking for vegetarian-friendly restaurants in Caelum.',
    city_id=DEST_2_ID,
)
display_restaurants(restaurants_2)

,restaurant_id,name,cuisine,avg_spend_per_person
1,restaurant_world_42_20260504_010845_0491,East End Vietnamese Bistro,,
2,restaurant_world_42_20260504_010845_0532,The West Quarter Brasserie,,
3,restaurant_world_42_20260504_010845_0544,Chez West Quarter,,
4,restaurant_world_42_20260504_010845_0620,Casa Cael,,
5,restaurant_world_42_20260504_010845_0520,Chinese House 5,,
6,restaurant_world_42_20260504_010845_0648,Italian House 8,,
7,restaurant_world_42_20260504_010845_0481,The East End Brasserie,,
8,restaurant_world_42_20260504_010845_0571,Indian House 7,,
9,restaurant_world_42_20260504_010845_0579,The Chinese Table,,
10,restaurant_world_42_20260504_010845_0574,The Northgate Brasserie,,


In [39]:
# Add Indian House 7
if restaurants_2:
    _resto_cael1 = restaurants_2[7]
    _day5 = str(_date.fromisoformat(START_DATE) + timedelta(days=4))
    session.add_activity_to_itinerary(
        date_str=_day5,
        city=DEST_2_ID,
        activity_id=_resto_cael1.get('restaurant_id', ''),
        activity_name=_resto_cael1.get('name', 'Restaurant'),
        location=_resto_cael1.get('location', DEST_2_ID),
        start_time='19:00',
        duration_hours=1.5,
        cost_usd=_resto_cael1.get('average_spend', 25.0) * PASSENGERS,
        category='restaurant',
        thought='Dinner at Indian House — best vegetarian-friendly option in Caelum.',
    )


  ✅ Dinner at Indian House — best vegetarian-friendly option in Caelum.


In [40]:
# Add Vietnamese Bistro
if restaurants_2:
    _resto_cael2 = restaurants_2[0]
    _day6 = str(_date.fromisoformat(START_DATE) + timedelta(days=5))
    session.add_activity_to_itinerary(
        date_str=_day6,
        city=DEST_2_ID,
        activity_id=_resto_cael2.get('restaurant_id', ''),
        activity_name=_resto_cael2.get('name', 'Restaurant'),
        location=_resto_cael2.get('location', DEST_2_ID),
        start_time='19:00',
        duration_hours=1.5,
        cost_usd=_resto_cael2.get('average_spend', 25.0) * PASSENGERS,
        category='restaurant',
        thought='Dinner at Vietnamese Bistro — vegetarian-friendly option in Caelum.',
    )


  ✅ Dinner at Vietnamese Bistro — vegetarian-friendly option in Caelum.


---
## 7. Events (Optional)

Book ticketed events that fall within the trip dates.

In [ ]:
events_1 = session.search_events(
    thought=f'Checking for events in {DEST_1_ID} during the trip.',
    city_id=DEST_1_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    max_price=session.budget_remaining * 0.1,  # events shouldn't blow the budget
)
display_events(events_1)

In [ ]:
# ── BOOK AN EVENT ─────────────────────────────────────────────────────────────
# Uncomment and edit if you want to book an event.

# if events_1:
#     chosen_event = events_1[0]
#     booking_event = session.book_event(
#         thought=f'Booking event: {chosen_event["name"]}.',
#         event_id=chosen_event['event_id'],
#         quantity=PASSENGERS,
#     )
#     if booking_event:
#         session.add_event_to_itinerary(
#             booking=booking_event,
#             city=DEST_1_ID,
#             event_name=chosen_event.get('name', ''),
#             thought='Added event booking to itinerary.',
#         )

print('Uncomment the block above to book an event.')

---
## 8. Free-form Planning Cells

Use these template cells to add any additional steps — extra searches, route
planning, additional activities, etc. Duplicate as needed.

In [ ]:
# ── TEMPLATE: call any tool ───────────────────────────────────────────────────
# result = session.search_attractions(
#     thought='...',
#     city_id=DEST_1_ID,
#     category='...',
# )
# display_attractions(result)

# ── TEMPLATE: add activity ────────────────────────────────────────────────────
# session.add_activity_to_itinerary(
#     date_str='YYYY-MM-DD',
#     city=DEST_1_ID,
#     activity_id='...',
#     activity_name='...',
#     start_time='10:00',
#     duration_hours=2.0,
#     cost_usd=0.0,
#     category='...',
#     thought='Reason for choosing this activity.',
# )

print('Template cell — add your own calls here.')

In [ ]:
# ── TEMPLATE: route planning ──────────────────────────────────────────────────
# routes_within = session.plan_route(
#     thought='Checking how to get from hotel to the hiking trail.',
#     origin_location_id='...',
#     destination_location_id='...',
#     departure_datetime=f'{START_DATE}T09:00:00',  # adjust time as needed
#     modes=['taxi', 'walk'],
# )
# print(routes_within)

print('Template cell — add your own route planning here.')

---
## 9. Progress Check

Run these at any time to inspect the current state of your plan.

In [45]:
session.show_itinerary()

════════════════════════════════════════════════════════════════
  ITINERARY  —  TEMPLATE
════════════════════════════════════════════════════════════════

  📅 2026-06-01  |  city_world_42_20260504_010845_0000  ($1096.10)
     ✈  FLIGHT: city_world_42_20260504_010845_0000 → city_world_42_20260504_010845_0001  09:00 → 12:10  $950.87  [FLT-50F91BE1]
     🏨  The Brindor Plaza  (★)  2026-06-01 → 2026-06-04  $145.23

  📅 2026-06-02  |  city_world_42_20260504_010845_0001  ($50.00)
     🎯  14:00  [museum]  Brindor Museum  $0.00
     🎯  10:00  [cultural_site]  Brindor Cultural Site  $0.00
     🎯  19:00  [restaurant]  Mediterranean Kitchen Bri3  $50.00

  📅 2026-06-03  |  city_world_42_20260504_010845_0001  ($208.20)
     🎯  09:00  [museum]  The Great Museum of Brindor  $0.00
     🎯  19:00  [restaurant]  Harbor View Vietnamese Bistro  $208.20
     🎯  14:00  [food_market]  Harbor View Food Market  $0.00

  📅 2026-06-04  |  city_world_42_20260504_010845_0001  ($1236.73)
     ✈  FLIGHT: city_world

In [46]:
session.show_budget()

  Budget            : $  5,000.00
  Transport         : $  3,087.38
  Accommodation     : $    382.23
  Activities/Events : $    612.34
  ─────────────────────────────
  Spent             : $  4,081.95
  Remaining         : $    918.05


In [47]:
session.show_trail()

════════════════════════════════════════════════════════════════
  EPISODE TRAIL  —  0759f9132644  (36 steps)
════════════════════════════════════════════════════════════════

  [00] [TOOL]  First I need to discover available cities and their IDs.
        Action: get_available_routes({})
        Obs [✓]: [{'city_id': 'city_world_42_20260504_010845_0000', 'city_name': 'Aeloria', 'description': 'Aeloria i

  [01] [TOOL]  Searching for flights from city_world_42_20260504_010845_0000 to city_world_42_20260504_010845_0001 on 2026-06-01.
        Action: search_flights({'origin_city_id': 'city_world_42_20260504_010845_0000', 'destination_city_id': )
        Obs [✓]: [{'edge_id': 'edge_world_42_20260504_010845_0008', 'airline': 'ZenithAir', 'flight_number': 'ZE202',

  [02] [TOOL]  Taking the cheapest outbound flight on 2026-06-01.
        Action: select_flight({'edge_id': 'edge_world_42_20260504_010845_0008', 'origin_city_name': '', 'desti)
        Obs [✓]: {'booking_id': 'FLT-50F91BE1', 'edg

---
## 10. Finalize & Save

When you're satisfied with the itinerary, finalize and save the episode log.
Use `'HUMAN_INCOMPLETE'` if you stopped early.

In [48]:
episode_log = session.finalize(
    termination_reason='DONE_ITINERARY',   # or 'HUMAN_INCOMPLETE'
)


  Episode finalized — 36 steps, $4081.95 spent, hard_score=1.00


In [49]:
saved_path = session.save()
print(f'Episode log saved to: {saved_path}')

  Saved → /Users/kyledennis/Documents/COLUMBIA SPRING 2026/Generative AI /Project/optimized-llm-planning-memory/outputs/human_episodes/ep_0759f9132644.json
Episode log saved to: /Users/kyledennis/Documents/COLUMBIA SPRING 2026/Generative AI /Project/optimized-llm-planning-memory/outputs/human_episodes/ep_0759f9132644.json


---
## 11. Evaluation

Score the human plan using the same deterministic metrics applied to agent episodes.

In [50]:
from optimized_llm_planning_memory.evaluation.deterministic import DeterministicEvaluator

evaluator = DeterministicEvaluator()
scores = evaluator.score(episode_log, user_request)

display_scores(scores)

────────────────────────────────────────────────────
  EVALUATION SCORES
────────────────────────────────────────────────────

  Constraint satisfaction:
    hard_constraint_ratio                1.000  [██████████]
    soft_constraint_score                0.293  [███░░░░░░░]
    budget_adherence                     1.000  [██████████]

  Tool usage:
    tool_efficiency                      0.947  [█████████░]
    tool_failure_rate                    0.000  [░░░░░░░░░░]
    avg_tool_latency_ms                  3.184  
    steps_per_episode                   36.000  

  Trip quality:
    destination_coverage_ratio           0.000  [░░░░░░░░░░]
    accommodation_coverage_ratio         0.333  [███░░░░░░░]
    activity_density_score               0.571  [██████░░░░]
    rest_day_ratio                       1.000  [██████████]
    schedule_overlap_score               1.000  [██████████]
    intra_city_feasibility               1.000  [██████████]
    logical_consistency                  1.00

In [52]:
from optimized_llm_planning_memory.evaluation.deterministic import DeterministicEvaluator
from app.utils.data_loader import load_episode as _load_ep

_agent_episode = _load_ep(AGENT_EPISODE_ID)
_agent_scores = DeterministicEvaluator().score(_agent_episode, user_request)

In [56]:
# ── COMPARISON: Human vs Agent ──────────────────────────────────────────────
# Only runs if you used Option A (loaded from an agent episode).

if AGENT_EPISODE_ID:
    import pandas as _pd

    _human_rc = episode_log.reward_components
    _cmp = _pd.DataFrame({
        'Metric': [
            'hard_constraint_score',
            'soft_constraint_score',
            'tool_efficiency_score',
            'logical_consistency_score',
            'total_reward',
        ],
        'Human': [
            _human_rc.hard_constraint_score,
            _human_rc.soft_constraint_score,
            _human_rc.tool_efficiency_score,
            _human_rc.logical_consistency_score,
            _human_rc.total_reward,
        ],
        f'Agent ({_agent_episode.agent_mode})': [
            _agent_scores.get('hard_constraint_ratio', 0.0),
            _agent_scores.get('soft_constraint_score', 0.0),
            _agent_scores.get('tool_efficiency', 0.0),
            _agent_scores.get('logical_consistency', 0.0),
            0.0,  # total_reward not computed by DeterministicEvaluator
        ],
    }).set_index('Metric').round(3)
    _cmp['Delta (Human − Agent)'] = (_cmp['Human'] - _cmp[f'Agent ({_agent_episode.agent_mode})']).round(3)
    display(_cmp)
else:
    print('No agent episode loaded (Option B or C was used). Skipping comparison.')

,Human,Agent (compressor),Delta (Human − Agent)
Metric,,,
hard_constraint_score,1.000,0.500,0.500
soft_constraint_score,0.293,0.400,-0.107
tool_efficiency_score,0.947,0.571,0.376
logical_consistency_score,1.000,1.000,0.000
total_reward,0.751,0.000,0.751


In [57]:
# Detailed constraint satisfaction breakdown
from optimized_llm_planning_memory.core.constraints import ConstraintSatisfactionEngine

engine = ConstraintSatisfactionEngine()
all_constraints = list(user_request.hard_constraints) + list(user_request.soft_constraints)
results = engine.evaluate(episode_log.final_itinerary, all_constraints)

print('CONSTRAINT SATISFACTION DETAIL')
print('─' * 70)
for r in results:
    c = next((c for c in all_constraints if c.constraint_id == r.constraint_id), None)
    c_type = c.constraint_type.value.upper() if c else '?'
    icon = '✓' if r.satisfied else '✗'
    print(f'  {icon} [{c_type}]  score={r.score:.2f}  {r.constraint_id}')
    print(f'       {r.explanation}')

CONSTRAINT SATISFACTION DETAIL
──────────────────────────────────────────────────────────────────────
  ✓ [HARD]  score=1.00  hc_budget
       Total cost $4081.95 vs budget $5000.00.
  ✓ [HARD]  score=1.00  hc_dates
       Start '2026-06-01'=='2026-06-01': True. End '2026-06-07'=='2026-06-07': True.
  ✗ [SOFT]  score=0.29  sc_hotel
       2/7 nights have accommodation.
  ✗ [SOFT]  score=0.30  sc_food
       Preference 'vegetarian' not matched; partial credit.


---
## 12. Inspect the Episode Log

Verify the structure of the saved log matches the agent format.

In [58]:
print(f'episode_id      : {episode_log.episode_id}')
print(f'agent_mode      : {episode_log.agent_mode}')
print(f'world_seed      : {episode_log.world_seed}')
print(f'total_steps     : {episode_log.total_steps}')
print(f'success         : {episode_log.success}')
print(f'termination     : {episode_log.termination_reason}')
print(f'days planned    : {len(episode_log.final_itinerary.days)}')
print(f'total_cost_usd  : ${episode_log.final_itinerary.total_cost_usd:.2f}')
print(f'tool_stats      : {len(episode_log.tool_stats)} tools used')
print()
print('Reward components:')
rc = episode_log.reward_components
print(f'  hard_constraint_score   : {rc.hard_constraint_score:.3f}')
print(f'  soft_constraint_score   : {rc.soft_constraint_score:.3f}')
print(f'  tool_efficiency_score   : {rc.tool_efficiency_score:.3f}')
print(f'  logical_consistency     : {rc.logical_consistency_score:.3f}')
print(f'  total_reward            : {rc.total_reward:.3f}')

episode_id      : 0759f9132644
agent_mode      : human_baseline
world_seed      : 42
total_steps     : 36
success         : True
termination     : DONE_ITINERARY
days planned    : 7
total_cost_usd  : $4081.95
tool_stats      : 7 tools used

Reward components:
  hard_constraint_score   : 1.000
  soft_constraint_score   : 0.293
  tool_efficiency_score   : 0.947
  logical_consistency     : 1.000
  total_reward            : 0.751


In [59]:
# Print the full trajectory as text (same format the agent and compressor see)
print(episode_log.trajectory.to_text(include_itinerary_snapshots=False))

[Step 0]
Thought: First I need to discover available cities and their IDs.
Action: get_available_routes({})
Observation: [{"city_id": "city_world_42_20260504_010845_0000", "city_name": "Aeloria", "description": "Aeloria is a city in the Northern Europe region.", "vibe_summary": "Aeloria is a emerging city in Northern Europe, off-the-beaten-path by international visitors. The climate is four distinct seasons, making it considered very safe for travellers. The food scene leans toward Mexican and Thai cuisine, and the city is known for its vibrant exhibition and culture scene.", "dominant_cuisines": ["Mexican", "Thai", "American"], "dominant_attraction_categories": [], "dominant_event_categories": ["exhibition", "culture"]}, {"city_id": "city_world_42_20260504_010845_0001", "city_name": "Brindor", "description": "Brindor is a city in the Northern Europe region.", "vibe_summary": "Brindor is a mid-tier city in Northern Europe, moderately visited by international visitors. The climate is fo

In [60]:
# Load the saved JSON and verify it round-trips correctly
from optimized_llm_planning_memory.utils.episode_io import load_episode

reloaded = load_episode(saved_path)
assert reloaded.episode_id == episode_log.episode_id
assert reloaded.agent_mode == 'human_baseline'
print(f'Episode round-trip OK — {saved_path.name}')

Episode round-trip OK — ep_0759f9132644.json
